# LogiScan Stage 3 Retraining (22-Class Fallacy Classifier)

This notebook trains the **Fine-Grained Detector** using the latest unified dataset. 
It uses **DeBERTa-v3-XSmall** to achieve high accuracy while remaining light enough for CPU inference on machines with limited RAM (e.g., 8GB).

### Step 1: Install Dependencies

In [ ]:
!pip install transformers[torch] datasets accelerate scikit-learn tqdm

### Step 2: Ingest Data
Upload your `data/unified_training_data.json` to the Colab session or mount Google Drive.

In [ ]:
import json

import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

def load_data(path):
    with open(path) as f:
        data = json.load(f)

    texts = [d["text"] for d in data]
    label_list = sorted(list(set(d["fallacy"] for d in data)))
    label2id = {l: i for i, l in enumerate(label_list)}
    labels = [label2id[d["fallacy"]] for d in data]

    print(f"Loaded {len(texts)} samples across {len(label_list)} classes.")

    return train_test_split(
        texts, labels, test_size=0.15, random_state=42, stratify=labels
    ), label_list

### Step 3: Start Training

In [ ]:
# POINT TO YOUR DATA FILE
DATA_PATH = "unified_training_data.json"

(X_train, X_val, y_train, y_val), label_list = load_data(DATA_PATH)
num_labels = len(label_list)

model_name = "microsoft/deberta-v3-xsmall"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(DEVICE)

train_ds = FallacyDataset(X_train, y_train, tokenizer)
val_ds = FallacyDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32 if torch.cuda.is_available() else 8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64 if torch.cuda.is_available() else 16)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        out = model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE), labels=batch["label"].to(DEVICE))
        out.loss.backward()
        optimizer.step()
        scheduler.step()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE)).logits
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Val Macro-F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("stage3_production_fallacy")
        tokenizer.save_pretrained("stage3_production_fallacy")
        with open("stage3_production_fallacy/labels.json", "w") as f: json.dump(label_list, f)
        print("✅ Saved Best Model")

### Step 4: Download Model
Zip the `stage3_production_fallacy` folder and download it back to your local `models/` directory.

In [ ]:
!zip -r stage3_model.zip stage3_production_fallacy